# ALSM v2: Active Learning Sparse Measurement
## Single-Electron Pump Plateau Characterization

**Method**: Schoinas 2024 (Appl. Phys. Lett. 125, 124001)  
**Convention**: $\eta = \log_{10}(|\langle n \rangle - 1|)$ — more negative = better quantization

**Algorithm**:
1. Coarse scan ($N_\text{coarse}$ points, greedy nearest-neighbor path)
2. Active Learning loop: single k-NN (k=4) with neighbor variance as uncertainty proxy
3. 2D piecewise linear interpolation on fine grid
4. $\eta_E$ exponential extrapolation for plateau quality beyond noise floor

**Outputs** (reproducing paper Fig.1):
- (c) Variance heatmap at intermediate AL iteration + selected points
- (d) Sparse measured $\eta_M$ points after all AL iterations
- (e) Interpolated $\eta_M$ map with $\eta_\text{noise}$ contour
- (f) $\eta$ vs $V_\text{exit}$ line fit at fixed $V_\text{ent}$
- (g) 2D extrapolated $\eta_E(V_\text{exit}, V_\text{ent})$ map

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.interpolate import griddata
from scipy.optimize import curve_fit           # Schoinas η_E nonlinear fit (skill §2)
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
import os
import shutil
from datetime import datetime

try:
    import pyvisa
    PYVISA_AVAILABLE = True
except ImportError:
    PYVISA_AVAILABLE = False
    print("[WARN] pyvisa not available — hardware mode disabled")

## Configuration

In [ ]:
class ALSMConfig:
    """All parameters for an ALSM run."""

    # --- Gate voltage ranges ---
    V_exit_min = -0.5     # V_exit (exit gate) range
    V_exit_max = -0.22
    V_ent_min  = -0.5     # V_ent (entrance gate) range
    V_ent_max  = -0.22

    # --- Grid resolution ---
    n_exit = 200          # V_exit grid points (columns)
    n_ent  = 200          # V_ent grid points (rows)

    # --- AL parameters (Schoinas 2024 defaults) ---
    N_coarse     = 400    # Initial coarse scan points
    N_meas       = 60     # Measurement points per AL batch
    N_candidates = 100    # Candidate pool size per AL round
    N_AL         = 20     # Max AL iterations
    k_neighbors  = 4      # k for k-NN (paper: k=4)

    # --- Physical parameters ---
    e_charge = 1.602176634e-19   # elementary charge (C)
    f_drive  = 0.2e9             # pump drive frequency (Hz)
    amp_gain = 1e-9              # Ithaco preamp gain (A/V)

    # --- eta extrapolation thresholds ---
    eta_noise = -3.5      # noise floor: eta < this is noise-dominated
    eta_max   = -0.6      # plateau boundary: eta > this is outside plateau

    # --- Soft saturation for ML (skill §5 Path 2) ---
    I_sat_nA = 5.0        # saturation threshold in nA

    # --- Simulation ---
    sim_mode = True       # True: use test function, False: use hardware
    sim_func = None       # set to a simulation function

## Instrument Controller (Skill §2, §3, §4, §7, §8)

Channel assignment (**CRITICAL — skill §2.1**):
- GPIB 8 = **V_exit** (exit gate)
- GPIB 1 = **V_ent** (entrance gate)
- GPIB 19 = **DMM** (Keithley 2000)

In [ ]:
class InstrumentController:
    """Encapsulates all hardware state (skill §7).

    Channel wiring (skill §2.1 — DO NOT CHANGE):
        GPIB43::8  = V_exit (exit gate)
        GPIB43::1  = V_ent  (entrance gate)
        GPIB43::19 = DMM    (Keithley 2000)
    """

    MAX_STEP_V   = 0.004   # 4 mV max voltage step (skill §3)
    RAMP_SETTLE  = 0.01    # 10 ms between ramp steps
    MEAS_SETTLE  = 0.3     # 300 ms before DMM read
    DMM_RETRIES  = 3       # retry attempts (skill §8)
    DMM_TIMEOUT  = 10000   # 10 s timeout (skill §8)

    def __init__(self, config):
        self.config = config
        self.sim_mode = config.sim_mode
        self.gain = config.amp_gain           # A/V (skill §6)
        self.current_V_ent  = 0.0
        self.current_V_exit = 0.0
        self._total_meas = 0
        self.rm = None
        self.yoko_exit = None   # GPIB 8
        self.yoko_ent  = None   # GPIB 1
        self.dmm       = None   # GPIB 19

    # ── connect / disconnect ──────────────────────────────

    def connect(self):
        if self.sim_mode:
            print("[SIM] Simulation mode — no hardware connected")
            return
        if not PYVISA_AVAILABLE:
            raise RuntimeError("pyvisa not installed — cannot connect")

        self.rm = pyvisa.ResourceManager()
        print(self.rm.list_resources())

        # V_exit = GPIB 8  (skill §2.1)
        self.yoko_exit = self.rm.open_resource(
            "GPIB43::8::INSTR",
            write_termination='\n', read_termination='\n')
        # V_ent = GPIB 1   (skill §2.1)
        self.yoko_ent = self.rm.open_resource(
            "GPIB43::1::INSTR",
            write_termination='\n', read_termination='\n')
        # DMM = GPIB 19
        self.dmm = self.rm.open_resource(
            "GPIB43::19::INSTR",
            write_termination='\n', read_termination='\n')
        self.dmm.timeout = self.DMM_TIMEOUT

        # Initialize Yokogawa outputs
        self.yoko_exit.write("F1R5O1E")
        self.yoko_ent.write("F1R5O1E")

        # Set DMM NPLC — try multiple SCPI variants (skill §8)
        for cmd in [":SENS:VOLT:NPLC 1.0", "NPLC 1.0", ":NPLC 1.0"]:
            try:
                self.dmm.write(cmd)
                break
            except Exception:
                pass

        self.current_V_ent  = 0.0
        self.current_V_exit = 0.0
        print("[HW] Connected: V_exit=GPIB8, V_ent=GPIB1, DMM=GPIB19")

    def ramp_to_safe(self):
        """Ramp all channels to 0 V (skill §4)."""
        self.set_voltages(0.0, 0.0)
        print("[SAFE] All channels ramped to 0 V")

    def close(self):
        """Safe shutdown: ramp to 0 V then disconnect (skill §4)."""
        self.ramp_to_safe()
        if not self.sim_mode:
            for inst in [self.yoko_ent, self.yoko_exit, self.dmm]:
                if inst is not None:
                    try:
                        inst.close()
                    except Exception:
                        pass
            if self.rm is not None:
                self.rm.close()
        print("[CLOSED] All instruments disconnected")

    # ── voltage control ───────────────────────────────────

    def set_voltages(self, V_ent, V_exit):
        """Ramp both channels simultaneously in <= 4 mV steps (skill §3).

        Both channels are interpolated in the same number of steps
        so they move together at each sub-step.
        """
        target = np.array([V_ent, V_exit])
        start  = np.array([self.current_V_ent, self.current_V_exit])
        max_delta = np.max(np.abs(target - start))
        n_steps = max(1, int(np.ceil(max_delta / self.MAX_STEP_V)))

        if not self.sim_mode:
            for k in range(1, n_steps + 1):
                alpha = k / n_steps
                vk = start + alpha * (target - start)
                # Write both channels at each step (skill §3)
                self.yoko_ent.write(f'S{vk[0]:.6f}E')
                self.yoko_exit.write(f'S{vk[1]:.6f}E')
                time.sleep(self.RAMP_SETTLE)

        self.current_V_ent  = V_ent
        self.current_V_exit = V_exit

    # ── measurement ───────────────────────────────────────

    def measure(self, V_ent, V_exit):
        """Set voltages and read DMM.

        Returns RAW DMM voltage (skill §5 Path 1).
        Physical current: I[A] = raw_V * self.gain
        """
        self.set_voltages(V_ent, V_exit)
        self._total_meas += 1

        if self.sim_mode:
            return self.config.sim_func(V_ent, V_exit)

        time.sleep(self.MEAS_SETTLE)
        for attempt in range(self.DMM_RETRIES):
            try:
                return float(self.dmm.query("fetch?"))
            except Exception as e:
                if attempt < self.DMM_RETRIES - 1:
                    time.sleep(0.5)
                else:
                    print(f"[DMM] Failed after {self.DMM_RETRIES} attempts: {e}")
                    return np.nan

    # ── gain management (skill §6) ───────────────────────

    def set_gain(self, new_gain):
        """Set Ithaco gain with human confirmation (skill §6.3).
        Skips prompt in simulation mode."""
        if not self.sim_mode:
            print(f"\n*** MANUAL GAIN CHANGE REQUIRED ***")
            print(f"Set Ithaco dial to: {new_gain:.0e} A/V")
            entered = float(input("Confirm gain value: "))
            if abs(entered - new_gain) / new_gain > 0.01:
                raise RuntimeError("Gain mismatch — aborting")
            time.sleep(2.0)   # preamp settling
        self.gain = float(new_gain)

    # ── unit conversion helpers ───────────────────────────

    def raw_to_current_A(self, raw_V):
        """Convert raw DMM voltage to current in Amperes."""
        return raw_V * self.gain

    def raw_to_n(self, raw_V):
        """Convert raw DMM voltage to <n> = I / (e*f)."""
        I_A = self.raw_to_current_A(raw_V)
        ef = self.config.e_charge * self.config.f_drive
        return I_A / ef

    @property
    def total_measurements(self):
        return self._total_meas

## k-NN Active Learner (Schoinas 2024)

Single k-NN (k=4) with **neighbor variance** as uncertainty proxy.

- Prediction: inverse-distance weighted mean of k nearest neighbors
- Uncertainty: variance of k neighbors' measured values
- High variance = neighbors disagree = likely near plateau boundary = measure here

In [ ]:
class KNN_ActiveLearner:
    """Single k-NN regressor with neighbor variance for AL.

    Following Schoinas 2024 (APL 125, 124001):
    - k=4 nearest neighbors, inverse-distance weighted prediction
    - Neighbor variance as AL acquisition function (uncertainty proxy)
    """

    def __init__(self, k=4):
        self.k = k
        self.X_train = None
        self.y_train = None
        self._nn = None

    def fit(self, X, y):
        """Fit model with training data."""
        self.X_train = np.copy(X)
        self.y_train = np.copy(y)
        k_use = min(self.k, len(X))
        self._nn = NearestNeighbors(n_neighbors=k_use, algorithm='auto')
        self._nn.fit(X)

    def predict(self, X_test):
        """Predict at test points.

        Returns
        -------
        y_pred : (N,) inverse-distance weighted mean
        y_var  : (N,) variance of k neighbors' y-values (AL uncertainty)
        """
        k_use = min(self.k, len(self.X_train))
        distances, indices = self._nn.kneighbors(X_test, n_neighbors=k_use)
        y_neighbors = self.y_train[indices]       # (N_test, k)

        # Inverse-distance weighted prediction
        weights = 1.0 / (distances + 1e-10)
        weights = weights / weights.sum(axis=1, keepdims=True)
        y_pred = np.sum(weights * y_neighbors, axis=1)

        # Neighbor variance — AL uncertainty proxy
        y_var = np.var(y_neighbors, axis=1)

        return y_pred, y_var

    @property
    def n_train(self):
        return 0 if self.X_train is None else len(self.X_train)

## Simulation Test Functions

Pump-like test function for $\eta_E$ validation, plus generic functions for AL algorithm testing.

In [ ]:
def sim_pump_plateau(V_ent, V_exit):
    """Simulated pump n=1 plateau for eta_E testing.

    Returns <n> = I/(ef).  The plateau shape is a product of two
    sigmoids (loading × emission) whose position shifts with V_ent.

    V_exit range: ~ [-0.5, -0.22]
    V_ent  range: ~ [-0.5, -0.22]
    """
    # Plateau center and width parameters
    V_exit_center = -0.36
    V_ent_center  = -0.36
    width_exit = 0.015      # sigmoid steepness for V_exit
    width_ent  = 0.020      # plateau width dependence on V_ent

    # V_ent shifts the plateau position in V_exit
    shift = (V_ent - V_ent_center) * 0.4

    # Loading transition: n goes 0→1 as V_exit increases past threshold
    loading  = 1.0 / (1.0 + np.exp(-(V_exit - (V_exit_center - 0.06 + shift)) / width_exit))
    # Emission transition: n goes 1→0 as V_exit increases past upper threshold
    emission = 1.0 / (1.0 + np.exp( (V_exit - (V_exit_center + 0.06 + shift)) / width_exit))

    n = loading * emission

    # Add small noise to simulate measurement uncertainty
    noise = np.random.normal(0, 1e-6)
    return n + noise


def sim_generic(V_ent, V_exit, icase=6):
    """Generic test functions for AL algorithm testing (from QMT notebook)."""
    x, y = V_exit, V_ent   # use V_exit, V_ent directly
    if icase == 6:
        return np.cos(0.1 * x) * np.cos(0.2 * y)
    if icase == 7:
        return np.sin(0.1 * x) ** 2 + np.cos(0.2 * y) ** 2
    if icase == 8:
        return np.cos(0.1 * x + np.sin(0.2 * y))
    if icase == 9:
        return np.cos(0.1 * x) + np.sin(0.2 * y)
    return 0.0

## Path Optimization & File I/O

In [ ]:
def greedy_nearest_neighbor_order(points, start_point=None):
    """Order points by greedy nearest-neighbor to minimize total voltage travel.

    This replaces the paper's random-jump measurement order with a
    physically safe traversal that keeps voltage steps small.
    """
    n = len(points)
    if n <= 1:
        return np.arange(n)

    visited = np.zeros(n, dtype=bool)
    order   = np.zeros(n, dtype=int)

    if start_point is not None:
        dists = np.linalg.norm(points - start_point, axis=1)
        order[0] = np.argmin(dists)
    else:
        order[0] = 0
    visited[order[0]] = True

    for i in range(1, n):
        current = points[order[i - 1]]
        dists = np.linalg.norm(points - current, axis=1)
        dists[visited] = np.inf
        order[i] = np.argmin(dists)
        visited[order[i]] = True

    return order


def write_accum_header(filepath, config, controller):
    """Write metadata header to accumulation file (skill §6.4, §10)."""
    with open(filepath, 'w') as f:
        f.write(f"# ALSM measurement data\n")
        f.write(f"# date={datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"# gain={controller.gain:.0e} A/V\n")
        f.write(f"# ef={config.e_charge * config.f_drive:.6e} A\n")
        f.write(f"# V_exit_range=[{config.V_exit_min}, {config.V_exit_max}]\n")
        f.write(f"# V_ent_range=[{config.V_ent_min}, {config.V_ent_max}]\n")
        f.write(f"# columns: V_ent  V_exit  y_raw\n")


def append_to_accum(filepath, V_ent, V_exit, y_raw):
    """Append one measurement to accumulation file (raw value, skill §5 Path 1)."""
    with open(filepath, 'a') as f:
        f.write(f"{V_ent}  {V_exit}  {y_raw}\n")

## $\eta$ and $\eta_E$ Computation

Convention: $\eta = \log_{10}(|\langle n \rangle - 1|)$ — more negative = better quantization.

$\eta_E$ extrapolation uses the **nonlinear Schoinas model** (skill §2, §3 — linear fitting is prohibited):
$$\eta(V) = \log_{10}\!\left(10^{a_1 V + b_1} + 10^{a_2 V + b_2}\right)$$
- Left asymptote $a_1 < 0$ (loading side), right asymptote $a_2 > 0$ (emission side)
- $\eta_E^{\min} = a_1 V_\mathrm{opt} + b_1$ where $V_\mathrm{opt} = (b_2 - b_1)/(a_1 - a_2)$
- **Mathematically guaranteed**: $\eta_E^{\min} <$ any measured $\eta$ value

$\eta_\text{noise}$ is estimated as the **10th percentile of actual measured data** (skill §4).  
Must NOT use interpolated/GPR-smoothed data — smoothing attenuates noise and gives a falsely shallow floor.

In [ ]:
def find_eta_noise(y_train, gain, ef, eta_max=-0.6):
    """Estimate η_noise as 10th percentile of measured η (skill §4).

    MUST use actual measured data — NOT interpolated or GPR-smoothed data.
    Smoothing attenuates noise, giving a falsely shallow η_noise.

    Parameters
    ----------
    y_train : 1D array, raw DMM voltage measurements
    gain    : float, Ithaco preamp gain (A/V)
    ef      : float, e * f (A)
    eta_max : upper cutoff to exclude transition region (default -0.6)

    Returns
    -------
    eta_noise : float, 10th-percentile η from actual measurements
    """
    n_vals   = y_train * gain / ef
    eta_vals = np.log10(np.clip(np.abs(n_vals - 1.0), 1e-15, None))
    valid    = eta_vals[np.isfinite(eta_vals) & (eta_vals < eta_max)]
    if len(valid) < 3:
        return eta_max
    return float(np.percentile(valid, 10))


def compute_eta(n_map):
    """Compute η = log₁₀(|⟨n⟩ − 1|) from ⟨n⟩ map.

    Parameters
    ----------
    n_map : 2D array of ⟨n⟩ = I/(ef) values.

    Returns
    -------
    eta_map : 2D array, same shape. More negative = better quantization.
    """
    deviation = np.abs(n_map - 1.0)
    deviation = np.clip(deviation, 1e-15, None)    # avoid log(0)
    return np.log10(deviation)


def _schoinas_model(V, a1, b1, a2, b2):
    """Nonlinear Schoinas η model (skill §2).

    η(V) = log₁₀(10^(a₁V+b₁) + 10^(a₂V+b₂))

    Numerically stable log-sum-exp formulation:
      η = max(L, R) + log₁₀(10^(L−max) + 10^(R−max))

    Left asymptote:  η → a₁V + b₁  (a₁ < 0, loading side)
    Right asymptote: η → a₂V + b₂  (a₂ > 0, emission side)
    """
    L  = a1 * V + b1
    R  = a2 * V + b2
    mx = np.maximum(L, R)
    return mx + np.log10(10.0 ** (L - mx) + 10.0 ** (R - mx))


def extrapolate_eta_line(v_exit, eta_values, eta_noise=-3.5, eta_max=-0.6):
    """Extrapolate η_E using the nonlinear Schoinas model (skill §2, §3).

    The former double-linear-fit approach is PROHIBITED (skill §3):
    linear fits attenuate asymptote slopes and produce η_E < η_min in
    only 9% of cases (Monte Carlo over sparse sampling, 100 trials).

    Model:  η(V) = log₁₀(10^(a₁V+b₁) + 10^(a₂V+b₂))
    η_E^min = a₁·V_opt + b₁   where  V_opt = (b₂−b₁)/(a₁−a₂)
    Mathematically guaranteed: η_E^min < min(η_data)

    Parameters
    ----------
    v_exit     : 1D array of V_exit values
    eta_values : 1D array of η at each V_exit
    eta_noise  : kept for API compatibility; not used in fit selection
    eta_max    : upper cutoff — exclude points outside plateau (η > eta_max)

    Returns
    -------
    eta_E    : η_E^min (scalar or np.nan on failure)
    fit_info : dict with model parameters, curve, residual, asymptotes
    """
    # Fit region: below plateau boundary (skill §2 — no lower bound needed)
    fit_mask = np.isfinite(eta_values) & (eta_values < eta_max)
    if fit_mask.sum() < 4:
        return np.nan, {}

    V_fd   = v_exit[fit_mask]
    eta_fd = eta_values[fit_mask]

    # Initial estimates via polyfit on each side of the minimum
    i_cen  = np.argmin(eta_fd)
    V_cen  = V_fd[i_cen]
    left   = V_fd <= V_cen
    right  = V_fd >= V_cen

    sL_i, bL_i = (np.polyfit(V_fd[left],  eta_fd[left],  1)
                  if left.sum()  >= 2 else (-1.0, eta_fd.mean()))
    sR_i, bR_i = (np.polyfit(V_fd[right], eta_fd[right], 1)
                  if right.sum() >= 2 else ( 1.0, eta_fd.mean()))

    try:
        popt, _ = curve_fit(_schoinas_model, V_fd, eta_fd,
                            p0=[sL_i, bL_i, sR_i, bR_i], maxfev=10000)
    except RuntimeError:
        return np.nan, {}

    a1, b1, a2, b2 = popt

    # Validate: asymptote slopes must be physically correct
    if a1 >= 0 or a2 <= 0:
        return np.nan, {}

    # η_E^min at asymptote intersection (skill §2)
    V_opt  = (b2 - b1) / (a1 - a2)
    eta_E  = a1 * V_opt + b1

    # Model curve and RMS residual
    V_model   = np.linspace(v_exit.min(), v_exit.max(), 300)
    eta_model = _schoinas_model(V_model, *popt)
    rms       = float(np.sqrt(np.mean(
        (_schoinas_model(V_fd, *popt) - eta_fd) ** 2)))

    fit_info = {
        'v_opt': V_opt,   'eta_E': eta_E,
        'a1': a1, 'b1': b1, 'a2': a2, 'b2': b2,
        'V_model': V_model, 'eta_model': eta_model,
        'fit_mask': fit_mask, 'rms': rms,
        # Backward-compat aliases (used by extrapolate_eta_map, plot_fig1f)
        'v_cross': V_opt, 'slope_L': a1, 'intercept_L': b1,
        'slope_R': a2,    'intercept_R': b2,
    }
    return eta_E, fit_info


def extrapolate_eta_map(eta_map, v_exit_grid, v_ent_grid,
                        eta_noise=-3.5, eta_max=-0.6):
    """Compute 2D η_E(V_exit, V_ent) map via Schoinas extrapolation along V_exit
    for each V_ent value.

    Parameters
    ----------
    eta_map    : (n_ent, n_exit) array of η values
    v_exit_grid: (n_exit,) V_exit values
    v_ent_grid : (n_ent,) V_ent values

    Returns
    -------
    eta_E_map  : (n_ent, n_exit) asymptotic-envelope map
                 (each row = max(a₁V+b₁, a₂V+b₂) for that V_ent)
    eta_E_1d   : (n_ent,) η_E^min as function of V_ent
    fit_infos  : list of fit_info dicts, one per V_ent row
    """
    n_ent  = len(v_ent_grid)
    n_exit = len(v_exit_grid)
    eta_E_1d  = np.full(n_ent, np.nan)
    fit_infos = []

    for i in range(n_ent):
        # eta_map[i, :] = η along V_exit at fixed V_ent = v_ent_grid[i]
        eta_line = eta_map[i, :]
        eta_E, info = extrapolate_eta_line(
            v_exit_grid, eta_line, eta_noise=eta_noise, eta_max=eta_max)
        eta_E_1d[i] = eta_E
        fit_infos.append(info)

    # 2D map: asymptotic envelope max(a₁V+b₁, a₂V+b₂) per row
    eta_E_map = np.full((n_ent, n_exit), np.nan)
    for i in range(n_ent):
        info = fit_infos[i]
        if not info:
            continue
        line_L = info['a1'] * v_exit_grid + info['b1']
        line_R = info['a2'] * v_exit_grid + info['b2']
        eta_E_map[i, :] = np.maximum(line_L, line_R)

    return eta_E_map, eta_E_1d, fit_infos

## ALSM Driver

Combined algorithm:
1. **Coarse scan**: $N_\text{coarse}$ points on uniform sub-grid, greedy nearest-neighbor path
2. **AL loop**: k-NN variance → select top $N_\text{candidates}$ → sample $N_\text{meas}$ → measure → retrain
3. **Interpolation**: 2D piecewise linear (scipy `griddata`) on fine grid
4. **Extrapolation**: $\eta_E$ via nonlinear Schoinas model (skill §2 — double-linear fit is prohibited)

Meshgrid convention (skill §2.3):
```
rows  i  → V_ent   (d)
cols  j  → V_exit  (b)
```

In [ ]:
def driver_alsm(config, controller):
    """Main ALSM driver.

    Parameters
    ----------
    config     : ALSMConfig instance
    controller : InstrumentController instance (already connected)

    Returns
    -------
    results : dict with all outputs for visualization
    """

    # ── Setup ─────────────────────────────────────────────
    timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = f'output_ALSM_{timestamp}'
    os.makedirs(output_dir, exist_ok=True)
    accum_path = os.path.join(output_dir, 'accum.txt')
    write_accum_header(accum_path, config, controller)

    print(f"Output : {output_dir}")
    print(f"V_exit : [{config.V_exit_min}, {config.V_exit_max}]")
    print(f"V_ent  : [{config.V_ent_min}, {config.V_ent_max}]")
    print(f"Grid   : {config.n_ent} x {config.n_exit}")
    print(f"AL     : N_coarse={config.N_coarse}, N_meas={config.N_meas}, "
          f"N_AL={config.N_AL}, k={config.k_neighbors}")

    start_time = time.time()

    # ── Build fine grid (skill §2.3 meshgrid convention) ──
    # b = V_exit (columns), d = V_ent (rows)
    b = np.linspace(config.V_exit_min, config.V_exit_max, config.n_exit)
    d = np.linspace(config.V_ent_min,  config.V_ent_max,  config.n_ent)
    B, D = np.meshgrid(b, d)
    # grid_points[i] = (V_ent, V_exit) — row-major order
    # B[i,j] = b[j] = V_exit,  D[i,j] = d[i] = V_ent
    grid_points = np.column_stack([D.ravel(), B.ravel()])   # (n_total, 2)
    n_total = len(grid_points)

    measured_mask   = np.zeros(n_total, dtype=bool)
    measured_values = np.full(n_total, np.nan)

    X_list = []   # (V_ent, V_exit) pairs
    y_list = []   # raw measured values

    # ── Soft saturation for ML training (skill §5 Path 2) ─
    I_sat_raw = config.I_sat_nA * 1e-9 / controller.gain   # convert to raw DMM volts
    def soft_sat(y_raw):
        return I_sat_raw * np.tanh(y_raw / I_sat_raw)

    # ── Step 1: Coarse scan ───────────────────────────────
    n_side_ent  = int(np.ceil(np.sqrt(config.N_coarse * config.n_ent / config.n_exit)))
    n_side_exit = int(np.ceil(config.N_coarse / n_side_ent))
    idx_ent  = np.linspace(0, config.n_ent  - 1, n_side_ent,  dtype=int)
    idx_exit = np.linspace(0, config.n_exit - 1, n_side_exit, dtype=int)

    coarse_flat = []
    for i_ent in idx_ent:
        for j_exit in idx_exit:
            coarse_flat.append(i_ent * config.n_exit + j_exit)
    coarse_flat = np.array(coarse_flat)
    coarse_points = grid_points[coarse_flat]

    # Order by greedy nearest-neighbor from current position
    start_pos = np.array([controller.current_V_ent, controller.current_V_exit])
    order = greedy_nearest_neighbor_order(coarse_points, start_point=start_pos)
    coarse_flat   = coarse_flat[order]
    coarse_points = coarse_points[order]

    print(f"\n--- Coarse scan: {len(coarse_points)} points ---")
    for pt, idx in zip(coarse_points, coarse_flat):
        val = controller.measure(pt[0], pt[1])   # pt = (V_ent, V_exit)
        measured_mask[idx]   = True
        measured_values[idx] = val
        X_list.append(pt)
        y_list.append(val)
        append_to_accum(accum_path, pt[0], pt[1], val)

    X_train = np.array(X_list)
    y_train = np.array(y_list)
    print(f"Coarse done: {len(y_train)} points, "
          f"{time.time() - start_time:.1f}s")

    # ── Step 2: Initialize k-NN model ─────────────────────
    model = KNN_ActiveLearner(k=config.k_neighbors)
    # ML training uses soft-saturated values (skill §5 Path 2)
    model.fit(X_train, soft_sat(y_train))
    print(f"Model: k-NN (k={config.k_neighbors}), neighbor variance")

    # ── Step 3: Active Learning loop ──────────────────────
    al_history = []
    snapshot_round = 3   # save variance map at this round for Fig.1(c)
    snapshot_data  = {}

    for al_round in range(config.N_AL):
        # Predict on all grid points (for variance map)
        y_pred_all, y_var_all = model.predict(grid_points)

        # Unmeasured points only
        unmeasured_idx = np.where(~measured_mask)[0]
        if len(unmeasured_idx) == 0:
            print("All grid points measured!")
            break

        y_var_unmeasured = y_var_all[unmeasured_idx]
        max_var = np.max(y_var_unmeasured)

        # Select top N_candidates by variance
        N_cand = min(config.N_candidates, len(unmeasured_idx))
        N_meas = min(config.N_meas, N_cand)
        top_cand_local = np.argsort(y_var_unmeasured)[-N_cand:]

        # Sample N_meas from candidates (uniform, per paper)
        selected_local    = np.random.choice(top_cand_local, size=N_meas, replace=False)
        selected_grid_idx = unmeasured_idx[selected_local]
        selected_points   = grid_points[selected_grid_idx]

        # Save snapshot at round 3 for Fig.1(c) visualization
        if al_round + 1 == snapshot_round:
            snapshot_data = {
                'variance_map': y_var_all.reshape(config.n_ent, config.n_exit),
                'selected_points': selected_points.copy(),
                'round': al_round + 1,
            }

        # Order by greedy nearest-neighbor
        current_pos = np.array([controller.current_V_ent, controller.current_V_exit])
        path_order = greedy_nearest_neighbor_order(selected_points, start_point=current_pos)
        selected_points   = selected_points[path_order]
        selected_grid_idx = selected_grid_idx[path_order]

        # Measure batch
        for pt, idx in zip(selected_points, selected_grid_idx):
            val = controller.measure(pt[0], pt[1])
            measured_mask[idx]   = True
            measured_values[idx] = val
            X_list.append(pt)
            y_list.append(val)
            append_to_accum(accum_path, pt[0], pt[1], val)

        # Retrain
        X_train = np.array(X_list)
        y_train = np.array(y_list)
        model.fit(X_train, soft_sat(y_train))

        total_meas = int(np.sum(measured_mask))
        elapsed = time.time() - start_time
        al_history.append((al_round + 1, total_meas, max_var, elapsed))

        print(f"AL {al_round+1:3d}/{config.N_AL}: "
              f"+{N_meas} pts, total={total_meas}, "
              f"max_var={max_var:.4e}, {elapsed:.1f}s")

    # ── Step 4: 2D interpolation (paper method) ───────────
    print(f"\n--- Post-processing ---")
    measured_idx = np.where(measured_mask)[0]

    # Piecewise linear interpolation (paper: scipy griddata)
    interp_values = griddata(
        grid_points[measured_idx], measured_values[measured_idx],
        grid_points, method='linear')

    # Fill NaN outside convex hull with k-NN predictions
    y_pred_final, _ = model.predict(grid_points)
    nan_mask = np.isnan(interp_values)
    interp_values[nan_mask] = y_pred_final[nan_mask]

    # For measured points, always use actual values
    interp_values[measured_mask] = measured_values[measured_mask]

    # Reshape to 2D: (n_ent, n_exit) — rows=V_ent, cols=V_exit
    interp_map = interp_values.reshape(config.n_ent, config.n_exit)

    total_time = time.time() - start_time
    total_pts  = len(y_train)
    print(f"Total measurements: {total_pts}")
    print(f"Grid points: {n_total} ({config.n_ent}x{config.n_exit})")
    print(f"Reduction: {n_total / total_pts:.1f}x fewer measurements")
    print(f"Total time: {total_time:.1f}s")

    # ── Save results ──────────────────────────────────────
    np.savez(os.path.join(output_dir, 'results.npz'),
             v_exit_grid=b, v_ent_grid=d,
             interp_map=interp_map,
             X_train=X_train, y_train=y_train,
             measured_mask=measured_mask.reshape(config.n_ent, config.n_exit),
             measured_values=measured_values.reshape(config.n_ent, config.n_exit),
             al_history=np.array(al_history) if al_history else np.array([]),
             gain=controller.gain)

    results = {
        'interp_map': interp_map,
        'X_train': X_train,
        'y_train': y_train,
        'v_exit_grid': b,
        'v_ent_grid': d,
        'measured_mask': measured_mask.reshape(config.n_ent, config.n_exit),
        'al_history': al_history,
        'snapshot_data': snapshot_data,
        'output_dir': output_dir,
        'total_time': total_time,
        'config': config,
        'gain': controller.gain,
    }
    return results

## Visualization — Paper Fig.1 Reproduction

| Function | Paper Figure | Content |
|----------|-------------|---------|
| `plot_fig1c` | Fig.1(c) | Variance heatmap at AL round 3 + selected points |
| `plot_fig1d` | Fig.1(d) | Sparse measured $\eta_M$ points after all AL iterations |
| `plot_fig1e` | Fig.1(e) | Interpolated $\eta_M$ map with $\eta_\text{noise}$ contour |
| `plot_fig1f` | Fig.1(f) | $\eta$ vs $V_\text{exit}$ line fit at fixed $V_\text{ent}$ |
| `plot_fig1g` | Fig.1(g) | 2D extrapolated $\eta_E(V_\text{exit}, V_\text{ent})$ map |

Axis labels: $V_\text{exit}$, $V_\text{ent}$ (skill §11.1 — never $V_1$, $V_2$)

In [ ]:
def _extent(results):
    """Return [V_exit_min, V_exit_max, V_ent_min, V_ent_max] for imshow."""
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    return [b[0], b[-1], d[0], d[-1]]


def plot_fig1c(results):
    """Fig.1(c): Variance heatmap at intermediate AL iteration + selected points."""
    snap = results.get('snapshot_data', {})
    if not snap:
        print("[SKIP] No snapshot data — increase N_AL or change snapshot_round")
        return

    extent = _extent(results)
    fig, ax = plt.subplots(figsize=(8, 7))

    im = ax.imshow(snap['variance_map'], origin='lower', extent=extent,
                   aspect='auto', cmap='hot_r')
    plt.colorbar(im, ax=ax, label='k-NN variance')

    # Selected measurement points for next round (black dots, per paper)
    sel = snap['selected_points']
    ax.scatter(sel[:, 1], sel[:, 0], c='black', s=8, zorder=5,
               label=f"$N_\\mathrm{{meas}}={len(sel)}$ selected")

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    ax.set_title(f"Fig.1(c): k-NN variance at AL round {snap['round']}")
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1c_variance.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_fig1d(results, ef=None):
    """Fig.1(d): Sparse measured eta_M points after all AL iterations."""
    X = results['X_train']       # (N, 2) = (V_ent, V_exit)
    y = results['y_train']       # raw values
    extent = _extent(results)

    if ef is None:
        cfg = results['config']
        ef = cfg.e_charge * cfg.f_drive
    gain = results['gain']

    n_vals   = y * gain / ef
    eta_vals = np.log10(np.clip(np.abs(n_vals - 1.0), 1e-15, None))

    fig, ax = plt.subplots(figsize=(8, 7))
    sc = ax.scatter(X[:, 1], X[:, 0], c=eta_vals, cmap='RdYlGn_r',
                    s=4, vmin=-6, vmax=0)
    plt.colorbar(sc, ax=ax,
                 label='$\\eta_M = \\log_{10}(|\\langle n \\rangle - 1|)$')
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    ax.set_title(f"Fig.1(d): Sparse measured $\\eta_M$ ({len(X)} points)")
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1d_sparse_eta.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_fig1e(results, ef=None, eta_noise=-3.5):
    """Fig.1(e): Interpolated eta_M map with eta_noise contour."""
    interp_map = results['interp_map']
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    extent = _extent(results)

    if ef is None:
        cfg = results['config']
        ef = cfg.e_charge * cfg.f_drive
    gain = results['gain']

    # Convert interpolated map to <n> then eta
    n_map   = interp_map * gain / ef
    eta_map = compute_eta(n_map)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(eta_map, origin='lower', extent=extent,
                   aspect='auto', cmap='RdYlGn_r', vmin=-6, vmax=0)
    plt.colorbar(im, ax=ax,
                 label='$\\eta_M = \\log_{10}(|\\langle n \\rangle - 1|)$')

    # eta_noise contour — white dashed (inner noise-floor boundary)
    B, D = np.meshgrid(b, d)
    ax.contour(B, D, eta_map, levels=[eta_noise],
               colors='white', linestyles='--', linewidths=1.5)

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    ax.set_title(f"Fig.1(e): Interpolated $\\eta_M$ map "
                 f"(dashed = $\\eta_\\mathrm{{noise}}={eta_noise:.2f}$)")
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1e_interp_eta.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_map


def plot_fig1f(results, eta_map, v_ent_idx=None,
               eta_noise=-3.5, eta_max=-0.6):
    """Fig.1(f): eta vs V_exit at fixed V_ent — nonlinear Schoinas model fit.

    Visualization per skill §5:
    - Gray dots: all data points
    - Open red circles: fit points (eta < eta_max)
    - Red solid curve: Schoinas model η(V)
    - Red dashed lines: left/right asymptotes
    - Dark-red star: η_E^min marker at asymptote intersection
    - Gray dashed: η_noise line
    - Text box: RMS residual
    """
    b = results['v_exit_grid']
    d = results['v_ent_grid']

    # Auto-select V_ent row with the most negative eta (best plateau)
    if v_ent_idx is None:
        row_min_eta = np.nanmin(eta_map, axis=1)
        v_ent_idx   = int(np.nanargmin(row_min_eta))

    eta_line  = eta_map[v_ent_idx, :]
    v_ent_val = d[v_ent_idx]

    eta_E, info = extrapolate_eta_line(b, eta_line,
                                       eta_noise=eta_noise, eta_max=eta_max)

    fig, ax = plt.subplots(figsize=(10, 6))

    # All data points — small gray dots
    ax.plot(b, eta_line, '.', color='#aaaaaa', markersize=3,
            label='$\\eta_M$ data', zorder=2)

    if info:
        # Fit points — open red circles (skill §5)
        ax.plot(b[info['fit_mask']], eta_line[info['fit_mask']],
                'o', mfc='none', mec='red', markersize=5, linewidth=1.2,
                label='fit points', zorder=3)

        # Schoinas model curve — red solid (skill §5)
        ax.plot(info['V_model'], info['eta_model'], 'r-', linewidth=2,
                label=f"Schoinas model (RMS = {info['rms']:.3f})", zorder=4)

        # Asymptotes — red dashed (skill §5)
        v_asym = np.array([b[0], b[-1]])
        ax.plot(v_asym, info['a1'] * v_asym + info['b1'],
                'r--', linewidth=1.2, alpha=0.65,
                label=f"loading asymptote ($a_1={info['a1']:.1f}$)")
        ax.plot(v_asym, info['a2'] * v_asym + info['b2'],
                'r--', linewidth=1.2, alpha=0.65,
                label=f"emission asymptote ($a_2={info['a2']:.1f}$)")

        # η_E^min — dark-red star marker (skill §5)
        ax.plot(info['v_opt'], eta_E, '*', color='darkred', markersize=14,
                zorder=5, label=f"$\\eta_E^{{\\min}} = {eta_E:.2f}$")
        ax.axhline(eta_E, color='darkred', linestyle=':', linewidth=1, alpha=0.5)

        # RMS residual text box (skill §5)
        ax.text(0.02, 0.05, f"RMS residual = {info['rms']:.3f}",
                transform=ax.transAxes, fontsize=9, color='red',
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='lightyellow', alpha=0.85))

    # η_noise — gray dashed (skill §5)
    ax.axhline(eta_noise, color='gray', linestyle='--', linewidth=1.2,
               label=f"$\\eta_{{\\mathrm{{noise}}}} = {eta_noise:.2f}$")
    ax.axhline(eta_max, color='lightgray', linestyle='-.', linewidth=1,
               label=f"$\\eta_{{\\mathrm{{max}}}} = {eta_max}$ (fit cutoff)")

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$\\eta = \\log_{10}(|\\langle n \\rangle - 1|)$')
    ax.set_title(f"Fig.1(f): Schoinas $\\eta_E$ fit at "
                 f"$V_{{\\mathrm{{ent}}}} = {v_ent_val:.4f}$ V")
    ax.legend(fontsize=8, loc='upper right', ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(top=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1f_eta_fit.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_E, info


def plot_fig1g(results, eta_map, eta_noise=-3.5, eta_max=-0.6):
    """Fig.1(g): 2D extrapolated eta_E(V_exit, V_ent) map."""
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    extent = _extent(results)

    eta_E_map, eta_E_1d, fit_infos = extrapolate_eta_map(
        eta_map, b, d, eta_noise=eta_noise, eta_max=eta_max)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: 2D eta_E map
    ax = axes[0]
    im = ax.imshow(eta_E_map, origin='lower', extent=extent,
                   aspect='auto', cmap='RdYlGn_r', vmin=-8, vmax=0)
    plt.colorbar(im, ax=ax,
                 label='$\\eta_E = \\log_{10}(|\\langle n \\rangle - 1|)$')
    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    ax.set_title('Fig.1(g): Extrapolated $\\eta_E(V_{exit}, V_{ent})$')

    # Right: eta_E vs V_ent (1D summary) with optimum annotation
    ax = axes[1]
    valid = np.isfinite(eta_E_1d)
    if np.any(valid):
        ax.plot(d[valid], eta_E_1d[valid], 'b.-', markersize=4)
        best_idx = int(np.nanargmin(eta_E_1d))
        ax.axhline(eta_E_1d[best_idx], color='r', linestyle='--', alpha=0.5)
        ax.annotate(f"best $\\eta_E = {eta_E_1d[best_idx]:.2f}$\n"
                    f"at $V_{{\\mathrm{{ent}}}} = {d[best_idx]:.4f}$ V",
                    xy=(d[best_idx], eta_E_1d[best_idx]),
                    xytext=(0.55, 0.3), textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=10, color='red')
    ax.set_xlabel('$V_{ent}$ (V)')
    ax.set_ylabel('$\\eta_E$')
    ax.set_title('$\\eta_E$ vs $V_{ent}$')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1g_eta_E_map.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_E_map, eta_E_1d


def plot_al_convergence(results):
    """AL convergence: max variance vs round (supplementary)."""
    history = results['al_history']
    if not history:
        print("[SKIP] No AL history")
        return

    history = np.array(history)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(history[:, 0], history[:, 2], 'b.-')
    ax.set_xlabel('AL Round')
    ax.set_ylabel('Max k-NN variance')
    ax.set_title('AL Convergence')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'al_convergence.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_voltage_steps(results):
    """Voltage step size histogram (cf. paper Fig.2 inset)."""
    X = results['X_train']
    diffs = np.diff(X, axis=0)
    # X columns: (V_ent, V_exit)
    step_sizes = np.linalg.norm(diffs, axis=1) * 1000   # mV

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.plot(step_sizes, 'b-', alpha=0.5, linewidth=0.5)
    ax.set_xlabel('Measurement index')
    ax.set_ylabel('Step size (mV)')
    ax.set_title('Voltage step sizes')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.hist(step_sizes, bins=50, color='steelblue', edgecolor='black')
    ax.set_xlabel('Step size (mV)')
    ax.set_ylabel('Count')
    ax.set_title(f"mean={np.mean(step_sizes):.1f} mV, "
                 f"max={np.max(step_sizes):.1f} mV")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'voltage_steps.pdf'),
                bbox_inches='tight')
    plt.show()

## Run: Simulation Mode (Pump Plateau)

Test with `sim_pump_plateau` — returns $\langle n \rangle$ directly, so:
- `sim_func` returns $\langle n \rangle$ (dimensionless)
- In sim mode, `gain` and `ef` are set so that `raw * gain / ef = raw`  
  (i.e., raw value IS $\langle n \rangle$)

In [ ]:
# ── Configuration ──
cfg = ALSMConfig()
cfg.sim_mode = True
cfg.sim_func = sim_pump_plateau

# Pump voltage range
cfg.V_exit_min = -0.5
cfg.V_exit_max = -0.22
cfg.V_ent_min  = -0.5
cfg.V_ent_max  = -0.22
cfg.n_exit = 200
cfg.n_ent  = 200

# AL parameters (paper defaults)
cfg.N_coarse     = 400
cfg.N_meas       = 60
cfg.N_candidates = 100
cfg.N_AL         = 20
cfg.k_neighbors  = 4

# In simulation: sim_func returns <n> directly,
# so set gain and ef so that raw * gain / ef = raw
cfg.amp_gain = 1.0
cfg.e_charge = 1.0
cfg.f_drive  = 1.0

# eta thresholds
cfg.eta_noise = -3.5
cfg.eta_max   = -0.6

# Soft saturation: in sim mode raw values are <n> ~ [0, 1],
# so I_sat_raw = I_sat_nA * 1e-9 / gain must be >> 1.
# Set I_sat_nA = 5e9 so that I_sat_raw = 5.0 (>> max <n>)
# This effectively disables saturation for the well-behaved sim function.
cfg.I_sat_nA = 5.0e9

# ── Connect & Run ──
ctrl = InstrumentController(cfg)
ctrl.connect()

try:
    results = driver_alsm(cfg, ctrl)
finally:
    ctrl.close()

In [ ]:
# ── Fig.1 Visualization ──

# Estimate η_noise from actual measured data (skill §4 — 10th percentile)
# MUST use y_train (raw measurements), not the interpolated map
_cfg = results['config']
_ef  = _cfg.e_charge * _cfg.f_drive
eta_noise_data = find_eta_noise(results['y_train'], results['gain'], _ef)
eta_max_data   = _cfg.eta_max
print(f"η_noise (10th percentile of {len(results['y_train'])} measurements): "
      f"{eta_noise_data:.3f}")

# (c) Variance heatmap at AL round 3
plot_fig1c(results)

# (d) Sparse measured η_M
plot_fig1d(results)

# (e) Interpolated η_M map — returns eta_map used by (f) and (g)
eta_map = plot_fig1e(results, eta_noise=eta_noise_data)

# (f) η vs V_exit — Schoinas nonlinear fit (skill §2)
plot_fig1f(results, eta_map, eta_noise=eta_noise_data, eta_max=eta_max_data)

# (g) 2D extrapolated η_E map
eta_E_map, eta_E_1d = plot_fig1g(results, eta_map,
                                  eta_noise=eta_noise_data, eta_max=eta_max_data)

# Supplementary
plot_al_convergence(results)
plot_voltage_steps(results)

## Ground Truth Comparison (Simulation Only)

Compare ALSM interpolated map with full dense scan to validate accuracy.

In [ ]:
# ── Ground truth: full dense scan ──
b = results['v_exit_grid']
d = results['v_ent_grid']
B_full, D_full = np.meshgrid(b, d)

# Generate reference map (without noise for clean comparison)
ref_map = np.zeros_like(B_full)
for i in range(len(d)):
    for j in range(len(b)):
        # Use noiseless version
        V_ent, V_exit = d[i], b[j]
        V_exit_center = -0.36
        V_ent_center  = -0.36
        shift = (V_ent - V_ent_center) * 0.4
        loading  = 1.0 / (1.0 + np.exp(-(V_exit - (V_exit_center - 0.06 + shift)) / 0.015))
        emission = 1.0 / (1.0 + np.exp( (V_exit - (V_exit_center + 0.06 + shift)) / 0.015))
        ref_map[i, j] = loading * emission

# Comparison metrics
mse = np.mean((results['interp_map'] - ref_map) ** 2)
max_err = np.max(np.abs(results['interp_map'] - ref_map))
print(f"MSE vs ground truth: {mse:.2e}")
print(f"Max absolute error:  {max_err:.2e}")

extent = [b[0], b[-1], d[0], d[-1]]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
im = ax.imshow(ref_map, origin='lower', extent=extent, aspect='auto', cmap='viridis')
ax.set_title(f"Ground Truth ({cfg.n_ent * cfg.n_exit} pts)")
ax.set_xlabel('$V_{exit}$ (V)'); ax.set_ylabel('$V_{ent}$ (V)')
plt.colorbar(im, ax=ax, label='$\\langle n \\rangle$')

ax = axes[1]
im = ax.imshow(results['interp_map'], origin='lower', extent=extent, aspect='auto', cmap='viridis')
ax.set_title(f"ALSM ({len(results['X_train'])} pts)")
ax.set_xlabel('$V_{exit}$ (V)'); ax.set_ylabel('$V_{ent}$ (V)')
plt.colorbar(im, ax=ax, label='$\\langle n \\rangle$')

ax = axes[2]
diff = np.abs(results['interp_map'] - ref_map)
im = ax.imshow(diff, origin='lower', extent=extent, aspect='auto', cmap='Reds')
ax.set_title(f"|Difference| (MSE={mse:.2e})")
ax.set_xlabel('$V_{exit}$ (V)'); ax.set_ylabel('$V_{ent}$ (V)')
plt.colorbar(im, ax=ax, label='|error|')

plt.tight_layout()
plt.savefig(os.path.join(results['output_dir'], 'ground_truth_comparison.pdf'),
            bbox_inches='tight')
plt.show()

## Run: Hardware Mode

Uncomment and configure for real pump measurements.  
**CRITICAL**: Verify GPIB addresses and Ithaco gain before running.

In [ ]:
# # ── Hardware Configuration ──
# cfg_hw = ALSMConfig()
# cfg_hw.sim_mode = False
#
# # Voltage range for pump characterization
# cfg_hw.V_exit_min = -0.5
# cfg_hw.V_exit_max = -0.22
# cfg_hw.V_ent_min  = -0.5
# cfg_hw.V_ent_max  = -0.22
# cfg_hw.n_exit = 200
# cfg_hw.n_ent  = 200
#
# # AL parameters (paper defaults)
# cfg_hw.N_coarse     = 400
# cfg_hw.N_meas       = 60
# cfg_hw.N_candidates = 100
# cfg_hw.N_AL         = 20
# cfg_hw.k_neighbors  = 4
#
# # Physical parameters
# cfg_hw.e_charge = 1.602176634e-19    # C
# cfg_hw.f_drive  = 0.2e9              # Hz
# cfg_hw.amp_gain = 1e-9               # A/V — MUST match Ithaco dial
#
# # eta_max threshold (eta_noise is estimated from data — do NOT set manually)
# cfg_hw.eta_max   = -0.6
#
# # Soft saturation
# cfg_hw.I_sat_nA = 5.0
#
# # ── Connect, Run, Shutdown ──
# ctrl_hw = InstrumentController(cfg_hw)
# ctrl_hw.connect()
#
# try:
#     results_hw = driver_alsm(cfg_hw, ctrl_hw)
#
#     # ── Visualize ──
#     # Estimate η_noise from actual measured data (skill §4)
#     _ef_hw = cfg_hw.e_charge * cfg_hw.f_drive
#     eta_noise_hw = find_eta_noise(results_hw['y_train'], results_hw['gain'], _ef_hw)
#     eta_max_hw   = cfg_hw.eta_max
#     print(f"η_noise (measured): {eta_noise_hw:.3f}")
#
#     plot_fig1c(results_hw)
#     plot_fig1d(results_hw)
#     eta_map_hw = plot_fig1e(results_hw, eta_noise=eta_noise_hw)
#     plot_fig1f(results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=eta_max_hw)
#     plot_fig1g(results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=eta_max_hw)
#     plot_al_convergence(results_hw)
#     plot_voltage_steps(results_hw)
# finally:
#     ctrl_hw.close()    # ALWAYS safe shutdown (skill §4)